In [1]:

from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# IMPORTS
# =========================================================
import os
import glob
import re
import pandas as pd
import numpy as np
import cv2
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.model_selection import KFold
from PIL import Image

# =========================================================
# MAIN PATH CONFIGURATION
# =========================================================
DATASET_ROOT = "/content/drive/MyDrive/Deep_learning_data"

LOCAL_SAVE_PATH = "/content/drive/MyDrive/Deep_learning_data/final_universal_light_direction_model.pth"
DRIVE_SAVE_PATH = os.path.join(DATASET_ROOT, "universal_light_direction_model.pth")

# Unified multi-dataset configuration including the 4 new real-world folders
ALL_DATASETS_CONFIG = {
    # Original Datasets
    "monkey": {
        "rgb_dir": "monkey_RGB_img",
        "depth_dir": "depth_monkey",
        "shadow_dir": "Shadow_monkey",
        "labels_file": "monkey_labels.csv"
    },
    "robot": {
        "rgb_dir": "robot_RGB_img",
        "depth_dir": "depth_robot",
        "shadow_dir": "Shadow_robot",
        "labels_file": "robot_labels.csv"
    },
    "snowman": {
        "rgb_dir": "new_snowman_RGB_img",
        "depth_dir": "depth_snowman",
        "shadow_dir": "Shadow_snow",
        "labels_file": "snowman_labels.csv"
    },
    # New Real-World Folders
    "flask": {
        "rgb_dir": "Flask/RGB",
        "depth_dir": "Flask/Depth",
        "shadow_dir": "Flask/Flask_clean_mask",
        "labels_file": "Flask/light_directions.xlsx"
    },
    "cup": {
        "rgb_dir": "cup/RGB",
        "depth_dir": "cup/Depth",
        "shadow_dir": "cup/cup_mask",
        "labels_file": "cup/light_directions.xlsx"
    },
    "handcream": {
        "rgb_dir": "Handcream/RGB",
        "depth_dir": "Handcream/Depth",
        "shadow_dir": "Handcream/hand_mask",
        "labels_file": "Handcream/light_directions.xlsx"
    },
    "lipstick": {
        "rgb_dir": "Lipstick/RGB",
        "depth_dir": "Lipstick/Depth",
        "shadow_dir": "Lipstick/crop_lipstick_shadow",
        "labels_file": "Lipstick/light_directions.xlsx"
    }
}

# =========================================================
# MODEL DEFINITION (5-Channel Input ResNet18)
# =========================================================
class ResNet18Encoder(nn.Module):
    def __init__(self, input_channels=5):
        super().__init__()
        resnet = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        original_conv = resnet.conv1

        self.conv1 = nn.Conv2d(
            input_channels, 64, kernel_size=original_conv.kernel_size,
            stride=original_conv.stride, padding=original_conv.padding, bias=False
        )

        with torch.no_grad():
            self.conv1.weight[:, :3] = original_conv.weight
            nn.init.kaiming_normal_(self.conv1.weight[:, 3:], mode='fan_out', nonlinearity='relu')

        self.bn1, self.relu, self.maxpool = resnet.bn1, resnet.relu, resnet.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4

    def forward(self, x):
        return self.layer4(self.layer3(self.layer2(self.layer1(self.maxpool(self.relu(self.bn1(self.conv1(x))))))))

class MultiModalRobustModel(nn.Module):
    def __init__(self, output_dim=3):
        super().__init__()
        self.encoder = ResNet18Encoder(input_channels=5)
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.flatten = nn.Flatten()
        self.fc_head = nn.Sequential(
            nn.Linear(512 * 16, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )

    def forward(self, rgb, depth, shadow):
        x = torch.cat([rgb, depth, shadow], dim=1)
        return F.normalize(self.fc_head(self.flatten(self.pool(self.encoder(x)))), p=2, dim=1)

# =========================================================
# UNIVERSAL BALANCED DATASET CLASS
# =========================================================
class UniversalMultiModalDataset(Dataset):
    def __init__(self, config_dict, train_mode=True):
        self.train_mode = train_mode
        self.all_samples = []

        print("====== Initializing Universal Dataset Pipeline ======")
        for domain_name, paths in config_dict.items():
            labels_p = os.path.normpath(os.path.join(DATASET_ROOT, paths["labels_file"]))
            if not os.path.exists(labels_p):
                print(f"Skipping domain '{domain_name}': File not found at {labels_p}")
                continue

            # Load the file
            if labels_p.endswith('.xlsx') or labels_p.endswith('.xls'):
                df = pd.read_excel(labels_p)
            else:
                df = pd.read_csv(labels_p)

            # Clean column strings immediately to eliminate casing and whitespace mismatches
            df.columns = [str(c).strip().lower() for c in df.columns]

            # Dynamic check for filename columns
            possible_file_cols = ['filename', 'file_name', 'image', 'img', 'name', 'image_name', 'imagename']
            file_col = next((c for c in possible_file_cols if c in df.columns), df.columns[0])

            # Dynamic check for vector targets
            light_cols = None
            possible_triplets = [
                ['light_x', 'light_y', 'light_z'],
                ['l_x', 'l_y', 'l_z'],
                ['light_dir_x', 'light_dir_y', 'light_dir_z'],
                ['light_direction_x', 'light_direction_y', 'light_direction_z']
            ]

            for triplet in possible_triplets:
                if all(c in df.columns for c in triplet):
                    light_cols = triplet
                    break

            if light_cols is None:
                # Fallback to the last 3 columns if nothing matches standard names
                light_cols = list(df.columns[-3:])

            domain_count = 0
            for _, row in df.iterrows():
                self.all_samples.append({
                    "domain": domain_name,
                    "filename": str(row[file_col]).strip(),
                    "target": [float(row[c]) for c in light_cols],
                    "paths": paths
                })
                domain_count += 1
            print(f"Loaded {domain_count} samples from Domain: [{domain_name}]")

        print(f"Combined Dataset Construction Completed. Total Mixed Samples: {len(self.all_samples)}\n")

    def denoise_shadow(self, pil_img):
        cv_img = np.array(pil_img)
        denoised = cv2.medianBlur(cv_img, 5)
        _, thresh = cv2.threshold(denoised, 127, 255, cv2.THRESH_BINARY)
        kernel = np.ones((3, 3), np.uint8)
        return Image.fromarray(cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel))

    def find_image_path(self, folder, filename):
        """ Robust look-up covering extension variations and zero-padding mismatches """
        exact = os.path.join(folder, filename)
        if os.path.exists(exact):
            return exact

        # Extract raw number pattern and extension context
        base_name = os.path.splitext(filename)[0]
        num_match = re.search(r'\d+', base_name)
        num_str = num_match.group(0) if num_match else None

        # Build list of alternative potential filenames
        candidates = [filename, base_name]
        for ext in ['jpg', 'jpeg', 'png', 'JPG', 'JPEG', 'PNG']:
            candidates.append(f"{base_name}.{ext}")
            if num_str:
                # Account for numerical zero-padding mismatches (e.g., frame_58 vs frame_0058)
                clean_base = base_name.replace(num_str, str(int(num_str)))
                candidates.append(f"{clean_base}.{ext}")
                candidates.append(f"{clean_base.replace(str(int(num_str)), f'{int(num_str):04d}')}.{ext}")

        # Test against calculated options
        for cand in set(candidates):
            cand_path = os.path.join(folder, cand)
            if os.path.exists(cand_path):
                return cand_path

        # Hard directory fallback scanning if os.path.exists() is missing Drive's virtual sync
        if os.path.exists(folder):
            all_files = os.listdir(folder)
            # Try finding a filename that contains our base name ignoring casing
            for f in all_files:
                if base_name.lower() in f.lower():
                    return os.path.join(folder, f)
            # Try finding a file based on index numbers
            if num_str:
                for f in all_files:
                    f_num = re.search(r'\d+', f)
                    if f_num and int(f_num.group(0)) == int(num_str):
                        return os.path.join(folder, f)

        raise FileNotFoundError(f"Missing sample file: {filename} inside folder context: {folder}")

    def __len__(self):
        return len(self.all_samples)

    def __getitem__(self, idx):
        sample = self.all_samples[idx]
        domain = sample["domain"]
        filename = sample["filename"]
        paths = sample["paths"]

        rgb_dir = os.path.normpath(os.path.join(DATASET_ROOT, paths["rgb_dir"]))
        depth_dir = os.path.normpath(os.path.join(DATASET_ROOT, paths["depth_dir"]))
        shadow_dir = os.path.normpath(os.path.join(DATASET_ROOT, paths["shadow_dir"]))

        rgb_p = self.find_image_path(rgb_dir, filename)
        depth_p = self.find_image_path(depth_dir, filename)
        shadow_p = self.find_image_path(shadow_dir, filename)

        rgb_img = Image.open(rgb_p).convert('RGB').resize((224, 224))
        depth_img = Image.open(depth_p).convert('L').resize((224, 224))
        shadow_img = self.denoise_shadow(Image.open(shadow_p).convert('L')).resize((224, 224))

        target_coords = list(sample["target"])

        # Geometrically aligned augmentations across separate channels
        if self.train_mode:
            # 1. Aligned Random Horizontal Flipping
            if np.random.rand() > 0.5:
                rgb_img = TF.hflip(rgb_img)
                depth_img = TF.hflip(depth_img)
                shadow_img = TF.hflip(shadow_img)
                # Flip target x coordinate to maintain physical space validity
                target_coords[0] = -target_coords[0]

            # 2. Aligned Random Rotation (-20 to +20 degrees)
            if np.random.rand() > 0.5:
                angle = float(np.random.uniform(-20, 20))
                rgb_img = TF.rotate(rgb_img, angle)
                depth_img = TF.rotate(depth_img, angle)
                shadow_img = TF.rotate(shadow_img, angle)

            # 3. Visual Domain Color Jittering (RGB Channel exclusive)
            if np.random.rand() > 0.5:
                color_jitter = transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)
                rgb_img = color_jitter(rgb_img)

        # Normalization steps matching preprocessing requirements
        rgb_tensor = TF.to_tensor(rgb_img)
        rgb_tensor = TF.normalize(rgb_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        depth_tensor = TF.to_tensor(depth_img)
        depth_tensor = TF.normalize(depth_tensor, mean=[0.5], std=[0.225])

        shadow_tensor = TF.to_tensor(shadow_img)
        shadow_tensor = TF.normalize(shadow_tensor, mean=[0.5], std=[0.225])

        target_tensor = torch.tensor(target_coords, dtype=torch.float32)

        return rgb_tensor, depth_tensor, shadow_tensor, target_tensor, domain

# =========================================================
# ENGINE TRAINING LOOPS (Balanced & Domain-Tracked)
# =========================================================
def run_universal_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Execution Target Device: {device}")

    full_dataset = UniversalMultiModalDataset(ALL_DATASETS_CONFIG, train_mode=True)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    best_overall_val_loss = float('inf')

    for fold, (train_idx, val_idx) in enumerate(kf.split(full_dataset)):
        print(f"\n==========================================")
        print(f"       STARTING MIXED CV FOLD {fold+1}/5        ")
        print(f"==========================================")

        train_sub = torch.utils.data.Subset(full_dataset, train_idx)
        val_sub = torch.utils.data.Subset(full_dataset, val_idx)

        # Toggle augmentation environments cleanly during execution phases
        train_sub.dataset.train_mode = True
        val_sub.dataset.train_mode = False

        # Randomize allocation dynamically to ensure mixed-batch variations
        train_loader = DataLoader(train_sub, batch_size=16, shuffle=True, num_workers=0, drop_last=True)
        val_loader = DataLoader(val_sub, batch_size=16, shuffle=False, num_workers=0)

        model = MultiModalRobustModel(output_dim=3).to(device)

        # Lowered robust learning rate boundary
        optimizer = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
        criterion = nn.MSELoss()

        epochs = 12
        for epoch in range(epochs):
            model.train()
            running_loss = 0.0

            for rgb, depth, shadow, targets, _ in train_loader:
                rgb, depth, shadow, targets = rgb.to(device), depth.to(device), shadow.to(device), targets.to(device)

                optimizer.zero_grad()
                outputs = model(rgb, depth, shadow)
                loss = criterion(outputs, targets)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                running_loss += loss.item()

            avg_train_loss = running_loss / len(train_loader)

            # Domain evaluation split step
            model.eval()
            domain_losses = {name: [] for name in ALL_DATASETS_CONFIG.keys()}
            total_val_loss = 0.0

            with torch.no_grad():
                for rgb, depth, shadow, targets, batch_domains in val_loader:
                    rgb, depth, shadow, targets = rgb.to(device), depth.to(device), shadow.to(device), targets.to(device)
                    preds = model(rgb, depth, shadow)

                    losses = torch.mean((preds - targets) ** 2, dim=1)
                    total_val_loss += losses.mean().item()

                    for i, d_name in enumerate(batch_domains):
                        if d_name in domain_losses:
                            domain_losses[d_name].append(losses[i].item())

            avg_val_loss = total_val_loss / len(val_loader)

            print(f"Fold {fold+1} | Epoch [{epoch+1:02d}/{epochs}] | Overall Train Loss: {avg_train_loss:.4f} | Overall Val Loss: {avg_val_loss:.4f}")
            domain_str = "   ↳ Val breakdown -> "
            for d_name, d_list in domain_losses.items():
                if d_list:
                    domain_str += f"{d_name}: {np.mean(d_list):.4f} | "
            print(domain_str[:-3]) # trim last pipe separator

        if avg_val_loss < best_overall_val_loss:
            best_overall_val_loss = avg_val_loss
            torch.save(model.state_dict(), LOCAL_SAVE_PATH)
            print(f"--> Saved current best global model structure to local cache!")

            try:
                shutil.copy2(LOCAL_SAVE_PATH, DRIVE_SAVE_PATH)
                print(f"--> Successfully synchronized model out to permanent Drive location.")
            except Exception as e:
                print(f"--> Sync skipped due to volatile Drive write constraints.")

if __name__ == "__main__":
    run_universal_training()

Mounted at /content/drive
Execution Target Device: cuda
====== Initializing Universal Dataset Pipeline ======
Loaded 200 samples from Domain: [monkey]
Loaded 300 samples from Domain: [robot]
Loaded 300 samples from Domain: [snowman]
Loaded 100 samples from Domain: [flask]
Loaded 86 samples from Domain: [cup]
Loaded 91 samples from Domain: [handcream]
Loaded 100 samples from Domain: [lipstick]
Combined Dataset Construction Completed. Total Mixed Samples: 1177


       STARTING MIXED CV FOLD 1/5        
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 174MB/s]


Fold 1 | Epoch [01/12] | Overall Train Loss: 0.0835 | Overall Val Loss: 0.0209
   ↳ Val breakdown -> monkey: 0.0640 | robot: 0.0221 | snowman: 0.0107 | flask: 0.0051 | cup: 0.0047 | handcream: 0.0126 | lipstick: 0.0073
Fold 1 | Epoch [02/12] | Overall Train Loss: 0.0203 | Overall Val Loss: 0.0113
   ↳ Val breakdown -> monkey: 0.0328 | robot: 0.0071 | snowman: 0.0084 | flask: 0.0064 | cup: 0.0039 | handcream: 0.0074 | lipstick: 0.0091
Fold 1 | Epoch [03/12] | Overall Train Loss: 0.0106 | Overall Val Loss: 0.0070
   ↳ Val breakdown -> monkey: 0.0167 | robot: 0.0069 | snowman: 0.0059 | flask: 0.0032 | cup: 0.0027 | handcream: 0.0032 | lipstick: 0.0043
Fold 1 | Epoch [04/12] | Overall Train Loss: 0.0072 | Overall Val Loss: 0.0064
   ↳ Val breakdown -> monkey: 0.0170 | robot: 0.0060 | snowman: 0.0031 | flask: 0.0048 | cup: 0.0032 | handcream: 0.0031 | lipstick: 0.0044
Fold 1 | Epoch [05/12] | Overall Train Loss: 0.0058 | Overall Val Loss: 0.0065
   ↳ Val breakdown -> monkey: 0.0170 | robot:

In [3]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF
from PIL import Image

# =========================================================
# PATH CONFIGURATION
# =========================================================
DATASET_ROOT = "/content/drive/MyDrive/Deep_learning_data"
DRIVE_SAVE_PATH = os.path.join(DATASET_ROOT, "universal_light_direction_model.pth")

# Switched evaluation domain configurations to look at your primary trained pipeline
EVAL_DATASETS_CONFIG = {
    "sample_prediction": {
        "rgb_dir": "sample_prediction/RGB",
        "depth_dir": "sample_prediction/Depth",
        "mask_dir": "sample_prediction/mask",
        "labels_file": "sample_prediction/light_directions.xlsx"
    }
}

# =========================================================
# METRIC FUNCTION
# =========================================================
def calculate_angular_error(v1, v2):
    v1_norm = F.normalize(v1, p=2, dim=1)
    v2_norm = F.normalize(v2, p=2, dim=1)
    dot_product = torch.sum(v1_norm * v2_norm, dim=1).clamp(-1.0, 1.0)
    return torch.rad2deg(torch.acos(dot_product))

# =========================================================
# DATASET PIPELINE
# =========================================================
class UniversalMultiModalDataset(Dataset):
    def __init__(self, config_dict):
        self.all_samples = []

        for domain_name, paths in config_dict.items():
            labels_p = os.path.normpath(os.path.join(DATASET_ROOT, paths["labels_file"]))
            if not os.path.exists(labels_p):
                print(f"Skipping domain '{domain_name}': Reference targets file missing at {labels_p}")
                continue

            # Handles original training sheets (.xlsx)
            if labels_p.endswith('.xlsx') or labels_p.endswith('.xls'):
                df = pd.read_excel(labels_p)
            else:
                df = pd.read_csv(labels_p)

            df.columns = [str(c).strip().lower() for c in df.columns]

            possible_file_cols = ['filename', 'file_name', 'image', 'img', 'name', 'image_name']
            file_col = next((c for c in possible_file_cols if c in df.columns), df.columns[0])

            light_cols = None
            possible_triplets = [
                ['light_x', 'light_y', 'light_z'],
                ['light_dir_x', 'light_dir_y', 'light_dir_z']
            ]
            for triplet in possible_triplets:
                if all(c in df.columns for c in triplet):
                    light_cols = triplet
                    break
            if light_cols is None:
                light_cols = list(df.columns[-3:])

            for idx, row in df.iterrows():
                fname = str(row[file_col]).strip()
                self.all_samples.append({
                    "domain": domain_name,
                    "filename": fname,
                    "target": [float(row[c]) for c in light_cols],
                    "paths": paths
                })

    def __len__(self):
        return len(self.all_samples)

    def __getitem__(self, idx):
        sample = self.all_samples[idx]
        filename = sample["filename"]
        paths = sample["paths"]

        rgb_p = os.path.join(os.path.normpath(os.path.join(DATASET_ROOT, paths["rgb_dir"])), filename)
        depth_p = os.path.join(os.path.normpath(os.path.join(DATASET_ROOT, paths["depth_dir"])), filename)
        mask_p = os.path.join(os.path.normpath(os.path.join(DATASET_ROOT, paths["mask_dir"])), filename)

        # Extension fallbacks for depth map / masks splits
        for p_var in [depth_p, mask_p]:
            if not os.path.exists(p_var):
                base_name = os.path.splitext(filename)[0]
                for ext in ['.png', '.jpg', '.jpeg']:
                    if os.path.exists(os.path.join(os.path.dirname(p_var), base_name + ext)):
                        if p_var == depth_p: depth_p = os.path.join(os.path.dirname(p_var), base_name + ext)
                        if p_var == mask_p: mask_p = os.path.join(os.path.dirname(p_var), base_name + ext)
                        break

        if not os.path.exists(rgb_p):
            raise FileNotFoundError(f"Missing RGB source image file target at: {rgb_p}")

        # 1. RGB
        rgb_img = Image.open(rgb_p).convert('RGB').resize((224, 224))
        rgb_tensor = TF.to_tensor(rgb_img)
        rgb_tensor = TF.normalize(rgb_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        # 2. Depth
        if os.path.exists(depth_p):
            depth_img = Image.open(depth_p).convert('L').resize((224, 224))
            depth_tensor = TF.to_tensor(depth_img)
        else:
            depth_tensor = torch.zeros((1, 224, 224), dtype=torch.float32)

        # 3. Shadow Mask
        if os.path.exists(mask_p):
            mask_img = Image.open(mask_p).convert('L').resize((224, 224))
            mask_tensor = TF.to_tensor(mask_img)
        else:
            mask_tensor = torch.zeros((1, 224, 224), dtype=torch.float32)

        target_tensor = torch.tensor(sample["target"], dtype=torch.float32)
        return rgb_tensor, depth_tensor, mask_tensor, target_tensor, filename

# =========================================================
# RUNNER
# =========================================================
def run_trained_dataset_evaluation():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Target Execution Device: {device}")

    test_dataset = UniversalMultiModalDataset(EVAL_DATASETS_CONFIG)
    if len(test_dataset) == 0:
        print("Error: Could not locate your training dataset samples inside sample_prediction/ folder.")
        return

    try:
        model = MultiModalRobustModel(output_dim=3).to(device)
    except NameError:
        print("Error: Make sure your 'MultiModalRobustModel' class configuration block is declared first.")
        return

    if os.path.exists(DRIVE_SAVE_PATH):
        model.load_state_dict(torch.load(DRIVE_SAVE_PATH, map_location=device))
        print(f"Loaded trained baseline parameters from: {DRIVE_SAVE_PATH}")
    else:
        print(f"Weights parameters missing at path destination: {DRIVE_SAVE_PATH}")
        return

    model.eval()

    print("\n" + "="*105)
    print(f" {'IMAGE':<13} | {'GROUND TRUTH VECTOR (L2-Norm)':<30} | {'PREDICTED VECTOR (L2-Norm)':<30} | {'ANGULAR ERROR'}")
    print("="*105)

    # We assess the first available image frame entry from the trained distribution data splits
    with torch.no_grad():
        for idx in range(min(5, len(test_dataset))): # Prints top 5 samples to avoid massive output loops
            rgb, depth, shadow, target, filename = test_dataset[idx]

            rgb_in = rgb.unsqueeze(0).to(device)
            depth_in = depth.unsqueeze(0).to(device)
            shadow_in = shadow.unsqueeze(0).to(device)
            target_in = target.unsqueeze(0).to(device)

            predicted_tensor = model(rgb_in, depth_in, shadow_in)

            gt_normalized = F.normalize(target_in, p=2, dim=1).cpu().numpy()[0]
            pred_normalized = F.normalize(predicted_tensor, p=2, dim=1).cpu().numpy()[0]

            angular_error = calculate_angular_error(predicted_tensor, target_in).item()

            gt_str = f"[{gt_normalized[0]:.4f}, {gt_normalized[1]:.4f}, {gt_normalized[2]:.4f}]"
            pred_str = f"[{pred_normalized[0]:.4f}, {pred_normalized[1]:.4f}, {pred_normalized[2]:.4f}]"

            print(f" {filename:<13} | {gt_str:<30} | {pred_str:<30} | {angular_error:.2f}°")
    print("="*105 + "\n")

if __name__ == "__main__":
    run_trained_dataset_evaluation()

Target Execution Device: cuda
Loaded trained baseline parameters from: /content/drive/MyDrive/Deep_learning_data/universal_light_direction_model.pth

 IMAGE         | GROUND TRUTH VECTOR (L2-Norm)  | PREDICTED VECTOR (L2-Norm)     | ANGULAR ERROR
 Apple_1.jpg   | [0.0048, 0.0166, 0.9998]       | [-0.2093, 0.1722, 0.9626]      | 15.36°
 Apple_2.jpg   | [-0.0047, 0.0113, 0.9999]      | [-0.5301, 0.3099, 0.7893]      | 37.32°



In [4]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF
from PIL import Image

# =========================================================
# PATH CONFIGURATION
# =========================================================
DATASET_ROOT = "/content/drive/MyDrive/Deep_learning_data"
DRIVE_SAVE_PATH = os.path.join(DATASET_ROOT, "universal_light_direction_model.pth")

# Switched evaluation domain configurations to look at your primary trained pipeline
EVAL_DATASETS_CONFIG = {
    "sample_prediction": {
        "rgb_dir": "sample_prediction/RGB",
        "depth_dir": "sample_prediction/Depth",
        "mask_dir": "sample_prediction/mask",
        "labels_file": "sample_prediction/light_directions.xlsx"
    }
}

# =========================================================
# METRIC FUNCTION
# =========================================================
def calculate_angular_error(v1, v2):
    v1_norm = F.normalize(v1, p=2, dim=1)
    v2_norm = F.normalize(v2, p=2, dim=1)
    dot_product = torch.sum(v1_norm * v2_norm, dim=1).clamp(-1.0, 1.0)
    return torch.rad2deg(torch.acos(dot_product))

# =========================================================
# DATASET PIPELINE
# =========================================================
class UniversalMultiModalDataset(Dataset):
    def __init__(self, config_dict):
        self.all_samples = []

        for domain_name, paths in config_dict.items():
            labels_p = os.path.normpath(os.path.join(DATASET_ROOT, paths["labels_file"]))
            if not os.path.exists(labels_p):
                print(f"Skipping domain '{domain_name}': Reference targets file missing at {labels_p}")
                continue

            # Handles original training sheets (.xlsx)
            if labels_p.endswith('.xlsx') or labels_p.endswith('.xls'):
                df = pd.read_excel(labels_p)
            else:
                df = pd.read_csv(labels_p)

            df.columns = [str(c).strip().lower() for c in df.columns]

            possible_file_cols = ['filename', 'file_name', 'image', 'img', 'name', 'image_name']
            file_col = next((c for c in possible_file_cols if c in df.columns), df.columns[0])

            light_cols = None
            possible_triplets = [
                ['light_x', 'light_y', 'light_z'],
                ['light_dir_x', 'light_dir_y', 'light_dir_z']
            ]
            for triplet in possible_triplets:
                if all(c in df.columns for c in triplet):
                    light_cols = triplet
                    break
            if light_cols is None:
                light_cols = list(df.columns[-3:])

            for idx, row in df.iterrows():
                fname = str(row[file_col]).strip()
                self.all_samples.append({
                    "domain": domain_name,
                    "filename": fname,
                    "target": [float(row[c]) for c in light_cols],
                    "paths": paths
                })

    def __len__(self):
        return len(self.all_samples)

    def __getitem__(self, idx):
        sample = self.all_samples[idx]
        filename = sample["filename"]
        paths = sample["paths"]

        rgb_p = os.path.join(os.path.normpath(os.path.join(DATASET_ROOT, paths["rgb_dir"])), filename)
        depth_p = os.path.join(os.path.normpath(os.path.join(DATASET_ROOT, paths["depth_dir"])), filename)
        mask_p = os.path.join(os.path.normpath(os.path.join(DATASET_ROOT, paths["mask_dir"])), filename)

        # Extension fallbacks for depth map / masks splits
        for p_var in [depth_p, mask_p]:
            if not os.path.exists(p_var):
                base_name = os.path.splitext(filename)[0]
                for ext in ['.png', '.jpg', '.jpeg']:
                    if os.path.exists(os.path.join(os.path.dirname(p_var), base_name + ext)):
                        if p_var == depth_p: depth_p = os.path.join(os.path.dirname(p_var), base_name + ext)
                        if p_var == mask_p: mask_p = os.path.join(os.path.dirname(p_var), base_name + ext)
                        break

        if not os.path.exists(rgb_p):
            raise FileNotFoundError(f"Missing RGB source image file target at: {rgb_p}")

        # 1. RGB
        rgb_img = Image.open(rgb_p).convert('RGB').resize((224, 224))
        rgb_tensor = TF.to_tensor(rgb_img)
        rgb_tensor = TF.normalize(rgb_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        # 2. Depth
        if os.path.exists(depth_p):
            depth_img = Image.open(depth_p).convert('L').resize((224, 224))
            depth_tensor = TF.to_tensor(depth_img)
        else:
            depth_tensor = torch.zeros((1, 224, 224), dtype=torch.float32)

        # 3. Shadow Mask
        if os.path.exists(mask_p):
            mask_img = Image.open(mask_p).convert('L').resize((224, 224))
            mask_tensor = TF.to_tensor(mask_img)
        else:
            mask_tensor = torch.zeros((1, 224, 224), dtype=torch.float32)

        target_tensor = torch.tensor(sample["target"], dtype=torch.float32)
        return rgb_tensor, depth_tensor, mask_tensor, target_tensor, filename

# =========================================================
# RUNNER
# =========================================================
def run_trained_dataset_evaluation():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Target Execution Device: {device}")

    test_dataset = UniversalMultiModalDataset(EVAL_DATASETS_CONFIG)
    if len(test_dataset) == 0:
        print("Error: Could not locate your training dataset samples inside sample_prediction/ folder.")
        return

    try:
        model = MultiModalRobustModel(output_dim=3).to(device)
    except NameError:
        print("Error: Make sure your 'MultiModalRobustModel' class configuration block is declared first.")
        return

    if os.path.exists(DRIVE_SAVE_PATH):
        model.load_state_dict(torch.load(DRIVE_SAVE_PATH, map_location=device))
        print(f"Loaded trained baseline parameters from: {DRIVE_SAVE_PATH}")
    else:
        print(f"Weights parameters missing at path destination: {DRIVE_SAVE_PATH}")
        return

    model.eval()

    print("\n" + "="*105)
    print(f" {'IMAGE':<13} | {'GROUND TRUTH VECTOR (L2-Norm)':<30} | {'PREDICTED VECTOR (L2-Norm)':<30} | {'ANGULAR ERROR'}")
    print("="*105)

    # We assess the first available image frame entry from the trained distribution data splits
    with torch.no_grad():
        for idx in range(min(5, len(test_dataset))): # Prints top 5 samples to avoid massive output loops
            rgb, depth, shadow, target, filename = test_dataset[idx]

            rgb_in = rgb.unsqueeze(0).to(device)
            depth_in = depth.unsqueeze(0).to(device)
            shadow_in = shadow.unsqueeze(0).to(device)
            target_in = target.unsqueeze(0).to(device)

            predicted_tensor = model(rgb_in, depth_in, shadow_in)

            gt_normalized = F.normalize(target_in, p=2, dim=1).cpu().numpy()[0]
            pred_normalized = F.normalize(predicted_tensor, p=2, dim=1).cpu().numpy()[0]

            angular_error = calculate_angular_error(predicted_tensor, target_in).item()

            gt_str = f"[{gt_normalized[0]:.4f}, {gt_normalized[1]:.4f}, {gt_normalized[2]:.4f}]"
            pred_str = f"[{pred_normalized[0]:.4f}, {pred_normalized[1]:.4f}, {pred_normalized[2]:.4f}]"

            print(f" {filename:<13} | {gt_str:<30} | {pred_str:<30} | {angular_error:.2f}°")
    print("="*105 + "\n")

if __name__ == "__main__":
    run_trained_dataset_evaluation()

Target Execution Device: cuda
Loaded trained baseline parameters from: /content/drive/MyDrive/Deep_learning_data/universal_light_direction_model.pth

 IMAGE         | GROUND TRUTH VECTOR (L2-Norm)  | PREDICTED VECTOR (L2-Norm)     | ANGULAR ERROR
 cropped_frame_0000.jpg | [-0.8632, 0.0711, 0.4998]      | [-0.8349, 0.0527, 0.5479]      | 3.37°
 cropped_frame_0001.jpg | [-0.3303, -0.0022, 0.9439]     | [-0.3510, -0.0393, 0.9355]     | 2.48°
 cropped_frame_0002.jpg | [0.1956, -0.0756, 0.9778]      | [-0.0409, 0.0250, 0.9988]      | 14.81°
 cropped_frame_0003.jpg | [-0.6074, 0.0178, 0.7942]      | [-0.5182, -0.0915, 0.8504]     | 8.71°
 cropped_frame_0004.jpg | [-0.8314, 0.0400, 0.5543]      | [-0.8099, 0.0417, 0.5851]      | 2.16°



In [5]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF
from PIL import Image

# =========================================================
# PATH CONFIGURATION
# =========================================================
DATASET_ROOT = "/content/drive/MyDrive/Deep_learning_data"
DRIVE_SAVE_PATH = os.path.join(DATASET_ROOT, "universal_light_direction_model.pth")

EVAL_DATASETS_CONFIG = {
    "sample_prediction": {
        "rgb_dir": "sample_prediction/RGB",
        "depth_dir": "sample_prediction/Depth",
        "mask_dir": "sample_prediction/mask",
        "labels_file": "sample_prediction/light_directions.xlsx"
    }
}

# =========================================================
# METRIC FUNCTION
# =========================================================
def calculate_angular_error(v1, v2):
    v1_norm = F.normalize(v1, p=2, dim=1)
    v2_norm = F.normalize(v2, p=2, dim=1)
    dot_product = torch.sum(v1_norm * v2_norm, dim=1).clamp(-1.0, 1.0)
    return torch.rad2deg(torch.acos(dot_product))

# =========================================================
# DATASET PIPELINE
# =========================================================
class UniversalMultiModalDataset(Dataset):
    def __init__(self, config_dict):
        self.all_samples = []

        for domain_name, paths in config_dict.items():
            labels_p = os.path.normpath(os.path.join(DATASET_ROOT, paths["labels_file"]))
            if not os.path.exists(labels_p):
                print(f"Skipping domain '{domain_name}': Reference targets file missing at {labels_p}")
                continue

            if labels_p.endswith('.xlsx') or labels_p.endswith('.xls'):
                df = pd.read_excel(labels_p)
            else:
                df = pd.read_csv(labels_p)

            df.columns = [str(c).strip().lower() for c in df.columns]

            possible_file_cols = ['filename', 'file_name', 'image', 'img', 'name', 'image_name']
            file_col = next((c for c in possible_file_cols if c in df.columns), df.columns[0])

            light_cols = None
            possible_triplets = [
                ['light_x', 'light_y', 'light_z'],
                ['light_dir_x', 'light_dir_y', 'light_dir_z']
            ]
            for triplet in possible_triplets:
                if all(c in df.columns for c in triplet):
                    light_cols = triplet
                    break
            if light_cols is None:
                light_cols = list(df.columns[-3:])

            for idx, row in df.iterrows():
                fname = str(row[file_col]).strip()
                self.all_samples.append({
                    "domain": domain_name,
                    "filename": fname,
                    "target": [float(row[c]) for c in light_cols],
                    "paths": paths
                })

    def __len__(self):
        return len(self.all_samples)

    def __getitem__(self, idx):
        sample = self.all_samples[idx]
        filename = sample["filename"]
        paths = sample["paths"]

        # Save absolute target path directories to return them in output
        rgb_folder_path = os.path.normpath(os.path.join(DATASET_ROOT, paths["rgb_dir"]))
        rgb_p = os.path.join(rgb_folder_path, filename)
        depth_p = os.path.join(os.path.normpath(os.path.join(DATASET_ROOT, paths["depth_dir"])), filename)
        mask_p = os.path.join(os.path.normpath(os.path.join(DATASET_ROOT, paths["mask_dir"])), filename)

        for p_var in [depth_p, mask_p]:
            if not os.path.exists(p_var):
                base_name = os.path.splitext(filename)[0]
                for ext in ['.png', '.jpg', '.jpeg']:
                    if os.path.exists(os.path.join(os.path.dirname(p_var), base_name + ext)):
                        if p_var == depth_p: depth_p = os.path.join(os.path.dirname(p_var), base_name + ext)
                        if p_var == mask_p: mask_p = os.path.join(os.path.dirname(p_var), base_name + ext)
                        break

        if not os.path.exists(rgb_p):
            raise FileNotFoundError(f"Missing RGB source image file target at: {rgb_p}")

        # 1. RGB
        rgb_img = Image.open(rgb_p).convert('RGB').resize((224, 224))
        rgb_tensor = TF.to_tensor(rgb_img)
        rgb_tensor = TF.normalize(rgb_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        # 2. Depth
        if os.path.exists(depth_p):
            depth_img = Image.open(depth_p).convert('L').resize((224, 224))
            depth_tensor = TF.to_tensor(depth_img)
        else:
            depth_tensor = torch.zeros((1, 224, 224), dtype=torch.float32)

        # 3. Shadow Mask
        if os.path.exists(mask_p):
            mask_img = Image.open(mask_p).convert('L').resize((224, 224))
            mask_tensor = TF.to_tensor(mask_img)
        else:
            mask_tensor = torch.zeros((1, 224, 224), dtype=torch.float32)

        target_tensor = torch.tensor(sample["target"], dtype=torch.float32)
        return rgb_tensor, depth_tensor, mask_tensor, target_tensor, filename, paths["rgb_dir"]

# =========================================================
# RUNNER
# =========================================================
def run_trained_dataset_evaluation():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Target Execution Device: {device}")

    test_dataset = UniversalMultiModalDataset(EVAL_DATASETS_CONFIG)
    if len(test_dataset) == 0:
        print("Error: Could not locate dataset configurations.")
        return

    try:
        model = MultiModalRobustModel(output_dim=3).to(device)
    except NameError:
        print("Error: Make sure your 'MultiModalRobustModel' class block is loaded.")
        return

    if os.path.exists(DRIVE_SAVE_PATH):
        model.load_state_dict(torch.load(DRIVE_SAVE_PATH, map_location=device))
        print(f"Loaded trained baseline parameters from: {DRIVE_SAVE_PATH}")
    else:
        print(f"Weights parameters missing at path destination: {DRIVE_SAVE_PATH}")
        return

    model.eval()

    print("\n" + "="*135)
    print(f" {'IMAGE':<13} | {'SOURCE FOLDER':<23} | {'GROUND TRUTH VECTOR (L2-Norm)':<30} | {'PREDICTED VECTOR (L2-Norm)':<30} | {'ANGULAR ERROR'}")
    print("="*135)

    with torch.no_grad():
        for idx in range(min(5, len(test_dataset))):
            rgb, depth, shadow, target, filename, source_folder = test_dataset[idx]

            rgb_in = rgb.unsqueeze(0).to(device)
            depth_in = depth.unsqueeze(0).to(device)
            shadow_in = shadow.unsqueeze(0).to(device)
            target_in = target.unsqueeze(0).to(device)

            predicted_tensor = model(rgb_in, depth_in, shadow_in)

            gt_normalized = F.normalize(target_in, p=2, dim=1).cpu().numpy()[0]
            pred_normalized = F.normalize(predicted_tensor, p=2, dim=1).cpu().numpy()[0]

            angular_error = calculate_angular_error(predicted_tensor, target_in).item()

            gt_str = f"[{gt_normalized[0]:.4f}, {gt_normalized[1]:.4f}, {gt_normalized[2]:.4f}]"
            pred_str = f"[{pred_normalized[0]:.4f}, {pred_normalized[1]:.4f}, {pred_normalized[2]:.4f}]"

            print(f" {filename:<13} | {source_folder:<23} | {gt_str:<30} | {pred_str:<30} | {angular_error:.2f}°")
    print("="*135 + "\n")

if __name__ == "__main__":
    run_trained_dataset_evaluation()

Target Execution Device: cuda
Loaded trained baseline parameters from: /content/drive/MyDrive/Deep_learning_data/universal_light_direction_model.pth

 IMAGE         | SOURCE FOLDER           | GROUND TRUTH VECTOR (L2-Norm)  | PREDICTED VECTOR (L2-Norm)     | ANGULAR ERROR
 cropped_frame_0000.jpg | sample_prediction/RGB   | [-0.8632, 0.0711, 0.4998]      | [-0.8349, 0.0527, 0.5479]      | 3.37°
 cropped_frame_0001.jpg | sample_prediction/RGB   | [-0.3303, -0.0022, 0.9439]     | [-0.3510, -0.0393, 0.9355]     | 2.48°
 cropped_frame_0002.jpg | sample_prediction/RGB   | [0.1956, -0.0756, 0.9778]      | [-0.0409, 0.0250, 0.9988]      | 14.81°
 cropped_frame_0003.jpg | sample_prediction/RGB   | [-0.6074, 0.0178, 0.7942]      | [-0.5182, -0.0915, 0.8504]     | 8.71°
 cropped_frame_0004.jpg | sample_prediction/RGB   | [-0.8314, 0.0400, 0.5543]      | [-0.8099, 0.0417, 0.5851]      | 2.16°

